# 🧠 Robust Multimodal Sentiment Analysis
## Combatting Text Fragility via Adaptive Modality Balancing

**3-Stage Training Pipeline** with:
- Adaptive Curriculum Dropout (Novel)
- Corrected OGM-GE Gradient Modulation
- Inter-Modal Contrastive Alignment (InfoNCE)
- Learnable Gating Fusion

---
### Quick Start
1. Run **Section 1** (Setup) first
2. Run **Section 2** (Data) to download the dataset
3. Run **Sections 3-5** for training (Stage 1 → 2 → 3)
4. Run **Section 6** for ablation study
5. Run **Section 7** for robustness evaluation (KEY results)
6. Run **Section 8** for visualizations

## 1. 🔧 Environment Setup

In [2]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Go to Runtime > Change runtime type > GPU")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


In [5]:
# Clone the repository from GitHub
!git clone https://github.com/saimayithri/Multi-Modal-Sentiment-Analysis.git
%cd Multi-Modal-Sentiment-Analysis

import os
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

Cloning into 'Multi-Modal-Sentiment-Analysis'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 36 (delta 7), reused 36 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 36.37 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Multi-Modal-Sentiment-Analysis
Working directory: /content/Multi-Modal-Sentiment-Analysis
Files: ['MSA_Research_Colab.ipynb', 'requirements.txt', '.gitignore', 'datasets', 'modules', 'models', 'src', '.git', 'run']


## 2. 📦 Dataset Download

In [11]:
from google.colab import drive
import os
import shutil

# 1. Mount your Google Drive
drive.mount('/content/drive')

os.makedirs('data', exist_ok=True)

# 2. Since the folder is in "Shared with me", the easiest way to access it in Colab
# is to go to Google Drive, right-click the "Processed" folder, and click "Add shortcut to Drive".
# Assuming you added the shortcut to your main Drive, the path will be:
drive_folder = '/content/drive/MyDrive/My_MSA_Project/data/' # Corrected path to include /content/

print(f"\nLooking for files in: {drive_folder}")

try:
    # List contents of the drive folder for debugging
    if os.path.exists(drive_folder):
        print(f"Contents of {drive_folder}:")
        for item in os.listdir(drive_folder):
            print(f"  - {item}")
    else:
        print(f"⚠️ {drive_folder} does not exist. Please check the path and ensure the shortcut is created.")

    # Copy Unaligned Data and rename it for the next cell
    source_file = os.path.join(drive_folder, 'unaligned_50.pkl')
    destination_file = 'data/mosi_data.pkl'

    if os.path.exists(source_file):
        print(f"Copying {os.path.basename(source_file)} to {destination_file}...")
        shutil.copy(source_file, destination_file)
        print("✅ Data ready!")
    else:
        print(f"⚠️ {os.path.basename(source_file)} not found in {drive_folder}. Please ensure the file exists.")

except Exception as e:
    print(f"Error during file copy: {e}")

print("\nFiles in your local data/ folder:")
for f in os.listdir('data'):
    print(f"  {f}: {os.path.getsize(os.path.join('data', f)) / 1e6:.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Looking for files in: /content/drive/MyDrive/My_MSA_Project/data/
Contents of /content/drive/MyDrive/My_MSA_Project/data/:
  - unaligned_50.pkl
Copying unaligned_50.pkl to data/mosi_data.pkl...
✅ Data ready!

Files in your local data/ folder:
  mosi_data.pkl: 554.2 MB


In [13]:
# Verify dataset structure
import pickle
import numpy as np

with open('data/mosi_data.pkl', 'rb') as f:
    data = pickle.load(f)

for split in ['train', 'valid', 'test']:
    print(f"\n--- {split.upper()} ---")
    for key in data[split]:
        arr = data[split][key]
        # Check if the array is a numpy array before accessing .shape and .dtype
        if isinstance(arr, np.ndarray):
            print(f"  {key:20s}: shape={str(arr.shape):20s} dtype={arr.dtype}")
        else:
            # For lists or other types, print type and length
            print(f"  {key:20s}: type={type(arr).__name__:20s} len={len(arr)}")

# These lines assume 'text', 'audio', and 'vision' are numpy arrays
print(f"\nText dim: {data['train']['text'].shape[2]}")
print(f"Audio dim: {data['train']['audio'].shape[2]}")
print(f"Vision dim: {data['train']['vision'].shape[2]}")
print(f"Train samples: {len(data['train']['text'])}")
print(f"Valid samples: {len(data['valid']['text'])}")
print(f"Test samples: {len(data['test']['text'])}")


--- TRAIN ---
  raw_text            : shape=(1284,)              dtype=<U581
  audio               : shape=(1284, 375, 5)       dtype=float64
  vision              : shape=(1284, 500, 20)      dtype=float64
  id                  : type=list                 len=1284
  text                : shape=(1284, 50, 768)      dtype=float32
  text_bert           : shape=(1284, 3, 50)        dtype=int64
  audio_lengths       : type=list                 len=1284
  vision_lengths      : type=list                 len=1284
  annotations         : type=list                 len=1284
  classification_labels: shape=(1284,)              dtype=float64
  regression_labels   : shape=(1284,)              dtype=float64

--- VALID ---
  raw_text            : shape=(229,)               dtype=<U231
  audio               : shape=(229, 375, 5)        dtype=float64
  vision              : shape=(229, 500, 20)       dtype=float64
  id                  : type=list                 len=229
  text                : shape=(

## 3. 🎓 Stage 1: Train Text-Only Teacher

In [ ]:
# Install required packages, explicitly reinstalling numpy and pandas
!pip install numpy==1.26.0 pandas -r requirements.txt --upgrade

  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached gdown-4.7.1-py3-none-any.whl.metadata (4.4 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusolver-12.0.4.66-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
Using cached gdown-4.7.1-py3-none-any.whl (15 kB)
Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl (532.3 MB)
Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl (366.2 MB)
Using cached nvidia_cusolver-12.0.4.66-py3-none-manylinux_2_27_x86_64.whl (200.9 MB)
Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl (7.6 MB)
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2
  Attempting uninstall: torch
    Found existing in

In [17]:
import sys
import os

# Add the current directory (root of the cloned repo) to the Python path
# This is necessary for importing modules like 'datasets.dataloader'
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print(f"Current working directory '{os.getcwd()}' added to sys.path.")

Current working directory '/content/Multi-Modal-Sentiment-Analysis' added to sys.path.


In [19]:
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# Set PYTHONPATH to include the current directory so modules like 'datasets' can be found
!PYTHONPATH=. python run/train.py \
    --stage 1 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --num_epochs 40 \
    --batch_size 32 \
    --lr 1e-4 \
    --seed 42 \
    --log_dir logs

--- Starting Stage 1: Training Text-Only Teacher ---
Stage 1 | Epoch  1 | Loss: 1.9014 | Valid Acc-2: 0.5972 | F1: 0.4884
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  2 | Loss: 1.8384 | Valid Acc-2: 0.5741 | F1: 0.4187
Stage 1 | Epoch  3 | Loss: 1.8051 | Valid Acc-2: 0.5741 | F1: 0.4187
Stage 1 | Epoch  4 | Loss: 1.7557 | Valid Acc-2: 0.6250 | F1: 0.5759
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  5 | Loss: 1.6972 | Valid Acc-2: 0.7222 | F1: 0.7166
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  6 | Loss: 1.6322 | Valid Acc-2: 0.7731 | F1: 0.7744
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  7 | Loss: 1.5587 | Valid Acc-2: 0.7731 | F1: 0.7744
Stage 1 | Epoch  8 | Loss: 1.5233 | Valid Acc-2: 0.7824 | F1: 0.7834
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  9 | Loss: 1.4960 | Valid Acc-2: 0.7593 | F1: 0.7596
Stage 

## 4. 📡 Stage 2: Train Audio/Vision Students (Knowledge Distillation)

In [23]:
!git pull origin main


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 623 bytes | 311.00 KiB/s, done.
From https://github.com/saimayithri/Multi-Modal-Sentiment-Analysis
 * branch            main       -> FETCH_HEAD
   aad1cfc..a149df5  main       -> origin/main
Updating aad1cfc..a149df5
Fast-forward
 run/train.py | 15 ++++++++++++---
 1 file changed, 12 insertions(+), 3 deletions(-)


In [24]:
!PYTHONPATH=. python run/train.py \
    --stage 2 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --num_epochs 40 \
    --batch_size 32 \
    --lr 1e-4 \
    --seed 42 \
    --log_dir logs

--- Starting Stage 2: Training A/V encoders with Decoupled Updates ---
Teacher model loaded and frozen.
Stage 2 | Epoch  1 | A-Acc: 0.4259 | V-Acc: 0.4537 | Avg: 0.4398
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  2 | A-Acc: 0.4676 | V-Acc: 0.5648 | Avg: 0.5162
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  3 | A-Acc: 0.4861 | V-Acc: 0.5694 | Avg: 0.5278
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  4 | A-Acc: 0.5278 | V-Acc: 0.5741 | Avg: 0.5509
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  5 | A-Acc: 0.5741 | V-Acc: 0.5741 | Avg: 0.5741
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  6 | A-Acc: 0.5926 | V-Acc: 0.5741 | Avg: 0.5833
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  7 | A-Acc: 0.5741 | V

## 5. 🔥 Stage 3: Full Model Fine-tuning (All Novel Techniques)

In [28]:
# Full model with ALL novel techniques enabled
!PYTHONPATH=. python run/train.py \
    --stage 3 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --modality_dropout 0.15 \
    --use_adaptive_dropout \
    --use_ogm \
    --use_contrastive \
    --num_epochs 40 \
    --batch_size 32 \
    --seed 42 \
    --log_dir logs

--- Starting Stage 3: Fine-tuning Full Model with Learnable Gating ---
Loaded trained A/V student encoders.
Re-loaded pre-trained text teacher encoder.
Epoch  1 | Loss: 1.8925 | Gate: 4.7299 | Contr: 7.6469 | Acc-2: 0.7639 | Acc-7: 0.3013 | F1: 0.7649 | MAE: 1.0711
         Gates: T=0.408 A=0.333 V=0.259 | GradNorms: T=0.367 A=0.126 V=0.165 | TextDrop: 0.121
*** New Best Full Model! Acc-2=0.7639 Saving to models/robust_hybrid_best.pt ***
Epoch  2 | Loss: 1.8800 | Gate: 4.8209 | Contr: 7.5293 | Acc-2: 0.7731 | Acc-7: 0.3013 | F1: 0.7743 | MAE: 1.0667
         Gates: T=0.423 A=0.324 V=0.253 | GradNorms: T=0.316 A=0.122 V=0.157 | TextDrop: 0.122
*** New Best Full Model! Acc-2=0.7731 Saving to models/robust_hybrid_best.pt ***
Epoch  3 | Loss: 1.8762 | Gate: 4.8709 | Contr: 7.4183 | Acc-2: 0.7824 | Acc-7: 0.2926 | F1: 0.7836 | MAE: 1.0807
         Gates: T=0.431 A=0.322 V=0.247 | GradNorms: T=0.302 A=0.110 V=0.153 | TextDrop: 0.125
*** New Best Full Model! Acc-2=0.7824 Saving to models/robu

### 5b. Train a Text-Dominant Baseline (for comparison)
This trains Stage 3 **without** any of our techniques — creating a text-dominant model to compare against.

In [30]:
# Baseline: no dropout, no OGM, no contrastive
!PYTHONPATH=. python run/train.py \
    --stage 3 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --final_model_path models/baseline_text_dominant.pt \
    --num_epochs 40 \
    --batch_size 32 \
    --seed 42 \
    --log_dir logs

--- Starting Stage 3: Fine-tuning Full Model with Learnable Gating ---
Loaded trained A/V student encoders.
Re-loaded pre-trained text teacher encoder.
Epoch  1 | Loss: 1.8855 | Gate: 5.6406 | Contr: 0.0000 | Acc-2: 0.7593 | Acc-7: 0.2926 | F1: 0.7603 | MAE: 1.0851
         Gates: T=0.419 A=0.327 V=0.254 | GradNorms: T=0.181 A=0.069 V=0.104 | TextDrop: 0.000
*** New Best Full Model! Acc-2=0.7593 Saving to models/baseline_text_dominant.pt ***
Epoch  2 | Loss: 1.8745 | Gate: 5.5714 | Contr: 0.0000 | Acc-2: 0.7546 | Acc-7: 0.2882 | F1: 0.7556 | MAE: 1.1148
         Gates: T=0.436 A=0.319 V=0.246 | GradNorms: T=0.170 A=0.072 V=0.107 | TextDrop: 0.000
Epoch  3 | Loss: 1.8668 | Gate: 5.5648 | Contr: 0.0000 | Acc-2: 0.7778 | Acc-7: 0.2969 | F1: 0.7787 | MAE: 1.0720
         Gates: T=0.451 A=0.314 V=0.235 | GradNorms: T=0.159 A=0.069 V=0.110 | TextDrop: 0.000
*** New Best Full Model! Acc-2=0.7778 Saving to models/baseline_text_dominant.pt ***
Epoch  4 | Loss: 1.8563 | Gate: 5.5118 | Contr: 0.0

## 6. 🔬 Ablation Study (Modality Analysis)

In [31]:
!PYTHONPATH=. python run/ablation_study.py \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --final_model_path models/robust_hybrid_best.pt \
    --student_model_path models/students_distilled_aligned.pt


     COMPREHENSIVE ABLATION STUDY

--- Part 1: Modality Ablation on Final Model ---
Loading: models/robust_hybrid_best.pt
Loading unimodal heads from: models/students_distilled_aligned.pt
  Injected student classifier weights into evaluation heads.
                                                      
     MODALITY ABLATION RESULTS
         Condition  Acc-2  Acc-7     F1    MAE    Corr
Full Model (T+A+V) 0.7683 0.2988 0.7685 1.2223  0.4979
         Text Only 0.4223 0.1691 0.2507 1.7472 -0.0080
        Audio Only 0.4223 0.1516 0.2507 2.0357 -0.0690
       Vision Only 0.4223 0.1647 0.2507 1.7385  0.0000

--- Modality Dominance Analysis ---
  Fusion Synergy Gap:         +0.5161  (Goal: > 0, means fusion helps)
  Audio-Text Disparity Ratio: 1.1651   (Goal: → 1.0)
  Vision-Text Disparity Ratio:0.9950   (Goal: → 1.0)

--- Accuracy Drop (Full - Unimodal) ---
  Text contribution:   +0.3460
  Audio contribution:  +0.3460
  Vision contribution: +0.3460
  (More balanced drops = less dominance)


## 7. 🛡️ Robustness Evaluation (KEY RESULTS)
This is the **most important** section. It produces the evidence that our balanced model
is more robust than text-dominant models when text is corrupted.

In [33]:
# Test OUR model (balanced) under text corruption
!PYTHONPATH=. python run/robustness_test.py \
    --model_path models/robust_hybrid_best.pt \
    --model_name "Ours (Balanced)" \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --noise_types gaussian token_dropout \
    --output_dir figures


Loading model: models/robust_hybrid_best.pt

  NOISE SWEEP: gaussian — Model: Ours (Balanced)
  gaussian @ 0%: Acc-2=0.7683 | F1=0.7685 | MAE=1.2223
  gaussian @ 10%: Acc-2=0.7241 | F1=0.7243 | MAE=1.3343
  gaussian @ 20%: Acc-2=0.6982 | F1=0.6999 | MAE=1.4169
  gaussian @ 30%: Acc-2=0.6585 | F1=0.6606 | MAE=1.5230
  gaussian @ 50%: Acc-2=0.6479 | F1=0.6500 | MAE=1.5273
  gaussian @ 70%: Acc-2=0.6235 | F1=0.6253 | MAE=1.6030
  gaussian @ 100%: Acc-2=0.6067 | F1=0.6082 | MAE=1.6672

  NOISE SWEEP: token_dropout — Model: Ours (Balanced)
  token_dropout @ 0%: Acc-2=0.7683 | F1=0.7685 | MAE=1.2223
  token_dropout @ 10%: Acc-2=0.7302 | F1=0.7318 | MAE=1.3524
  token_dropout @ 20%: Acc-2=0.7134 | F1=0.7149 | MAE=1.4261
  token_dropout @ 30%: Acc-2=0.6692 | F1=0.6684 | MAE=1.5574
  token_dropout @ 50%: Acc-2=0.6113 | F1=0.5977 | MAE=1.7595
  token_dropout @ 70%: Acc-2=0.5183 | F1=0.4684 | MAE=2.0305
  token_dropout @ 100%: Acc-2=0.4223 | F1=0.2507 | MAE=2.3833

  MODALITY ABLATION: Text comp

In [35]:
# Test BASELINE model (text-dominant) under same corruption
!PYTHONPATH=. python run/robustness_test.py \
    --model_path models/baseline_text_dominant.pt \
    --model_name "Baseline (Text-Dominant)" \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --noise_types gaussian token_dropout \
    --output_dir figures


Loading model: models/baseline_text_dominant.pt

  NOISE SWEEP: gaussian — Model: Baseline (Text-Dominant)
  gaussian @ 0%: Acc-2=0.7683 | F1=0.7679 | MAE=1.1961
  gaussian @ 10%: Acc-2=0.7241 | F1=0.7242 | MAE=1.2728
  gaussian @ 20%: Acc-2=0.6890 | F1=0.6901 | MAE=1.4082
  gaussian @ 30%: Acc-2=0.6662 | F1=0.6679 | MAE=1.4462
  gaussian @ 50%: Acc-2=0.6585 | F1=0.6603 | MAE=1.5431
  gaussian @ 70%: Acc-2=0.5884 | F1=0.5902 | MAE=1.6735
  gaussian @ 100%: Acc-2=0.6067 | F1=0.6078 | MAE=1.6675

  NOISE SWEEP: token_dropout — Model: Baseline (Text-Dominant)
  token_dropout @ 0%: Acc-2=0.7683 | F1=0.7679 | MAE=1.1961
  token_dropout @ 10%: Acc-2=0.7485 | F1=0.7493 | MAE=1.2856
  token_dropout @ 20%: Acc-2=0.7149 | F1=0.7164 | MAE=1.3844
  token_dropout @ 30%: Acc-2=0.6555 | F1=0.6570 | MAE=1.5613
  token_dropout @ 50%: Acc-2=0.6433 | F1=0.6444 | MAE=1.6578
  token_dropout @ 70%: Acc-2=0.5915 | F1=0.5920 | MAE=1.7957
  token_dropout @ 100%: Acc-2=0.5198 | F1=0.5223 | MAE=2.0467

  MODALI

In [36]:
# Compare robustness side-by-side
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load results (you may need to load from separate files)
results_path = 'logs/robustness_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\n📊 Robustness Results Loaded")
    print(f"  Model: {results['model_name']}")
    print(f"  Text-Off Acc-2: {results['text_off_metrics']['acc2']:.4f}")
    print(f"  Text-Off F1:    {results['text_off_metrics']['f1']:.4f}")
else:
    print("Run robustness tests first!")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### 7b. Combined Robustness Comparison Plot

In [ ]:
# Side-by-side robustness comparison (run after both models are tested)
import sys
sys.path.append('.')
from run.robustness_test import (
    evaluate_with_corruption, add_gaussian_noise, token_dropout
)
from datasets.dataloader import getdataloader
from models.msamodel import MSAModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataloader, orig_dim = getdataloader('mosi', 32, 'data/mosi_data.pkl')

# Load both models
models = {}
for name, path in [('Ours (Balanced)', 'models/robust_hybrid_best.pt'),
                    ('Baseline (Text-Dominant)', 'models/baseline_text_dominant.pt')]:
    if os.path.exists(path):
        m = MSAModel(output_dim=7, orig_dim=orig_dim, proj_dim=40, layers=5).to(device)
        m.load_state_dict(torch.load(path, map_location=device), strict=False)
        m.eval()
        models[name] = m
        print(f"Loaded: {name}")

if len(models) == 2:
    levels = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {'Ours (Balanced)': '#2ECC71', 'Baseline (Text-Dominant)': '#E74C3C'}

    for noise_idx, (noise_name, noise_fn, param_name) in enumerate([
        ('Gaussian Noise', add_gaussian_noise, 'noise_ratio'),
        ('Token Dropout', token_dropout, 'drop_ratio')
    ]):
        ax = axes[noise_idx]
        for model_name, model in models.items():
            accs = []
            for level in levels:
                if level == 0:
                    m = evaluate_with_corruption(model, dataloader['test'], device)
                else:
                    m = evaluate_with_corruption(
                        model, dataloader['test'], device,
                        noise_fn, {param_name: level})
                accs.append(m['acc2'])
            ax.plot(levels, accs, '-o', color=colors[model_name],
                    linewidth=2.5, markersize=7, label=model_name)

        ax.set_title(f'Accuracy Under {noise_name}', fontweight='bold', fontsize=14)
        ax.set_xlabel('Corruption Level', fontsize=12)
        ax.set_ylabel('Binary Accuracy (Acc-2)', fontsize=12)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig('figures/robustness_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ This is the KEY figure for your paper!")
    print("A flatter green line = our model is more robust.")
else:
    print("Need both models trained. Run Stage 3 and the baseline cell above.")

## 8. 📊 Visualization Suite

In [ ]:
# Generate all training visualization plots
os.makedirs('figures', exist_ok=True)
!python run/visualize.py --log_dir logs --output_dir figures --stage 3 --seed 42

In [ ]:
# Display all generated figures inline
from IPython.display import Image, display
import glob

figures = sorted(glob.glob('figures/*.png'))
for fig_path in figures:
    print(f"\n{'='*60}")
    print(f"  {os.path.basename(fig_path)}")
    print(f"{'='*60}")
    display(Image(filename=fig_path, width=800))

In [ ]:
# t-SNE visualization (takes ~2 minutes)
!python run/visualize.py \
    --log_dir logs --output_dir figures \
    --tsne \
    --model_path models/robust_hybrid_best.pt \
    --data_path data/mosi_data.pkl

In [ ]:
if os.path.exists('figures/tsne_representations.png'):
    display(Image(filename='figures/tsne_representations.png', width=900))

## 9. 📋 Final Results Summary Table

In [ ]:
import pandas as pd
import json

# Compile all results into a publication-ready table
print("\n" + "="*80)
print("  FINAL RESULTS SUMMARY")
print("="*80)

# Load Stage 3 log
log_path = 'logs/training_log_stage3_seed42.json'
if os.path.exists(log_path):
    with open(log_path) as f:
        log = json.load(f)

    # Best epoch metrics
    best_epoch = max(log['epochs'], key=lambda e: e.get('val_acc2', 0))

    print(f"\n📈 Best Model (Epoch {best_epoch['epoch']}):")
    print(f"  Acc-2: {best_epoch.get('val_acc2', 'N/A')}")
    print(f"  Acc-7: {best_epoch.get('val_acc7', 'N/A')}")
    print(f"  F1:    {best_epoch.get('val_f1', 'N/A')}")
    print(f"  MAE:   {best_epoch.get('val_mae', 'N/A')}")
    print(f"  Corr:  {best_epoch.get('val_corr', 'N/A')}")

    # Final gate distribution
    final = log['epochs'][-1]
    if 'avg_gate_text' in final:
        print(f"\n⚖️ Final Gate Distribution:")
        print(f"  Text:   {final['avg_gate_text']:.4f}")
        print(f"  Audio:  {final['avg_gate_audio']:.4f}")
        print(f"  Vision: {final['avg_gate_vision']:.4f}")
        uniform = abs(final['avg_gate_text'] - 1/3) + abs(final['avg_gate_audio'] - 1/3) + abs(final['avg_gate_vision'] - 1/3)
        print(f"  Deviation from uniform: {uniform:.4f} (lower = more balanced)")
else:
    print("Run training first to see results.")

In [ ]:
# Generate LaTeX-ready results table for your paper
print("\n📝 LaTeX Table (copy-paste into your paper):")
print("")
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Performance comparison under text corruption on CMU-MOSI}")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r"Method & Clean Acc-2 & Noisy-30\% & Noisy-50\% & Text-Off \\")
print(r"\midrule")
print(r"Text-Dominant Baseline & -- & -- & -- & -- \\")
print(r"Ours (Balanced) & -- & -- & -- & -- \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\label{tab:robustness}")
print(r"\end{table}")
print("\n(Fill in the -- values from your robustness test results above)")

## 10. 💾 Save Everything to Google Drive

In [ ]:
# Mount Google Drive and save results
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/MSA_Results'
os.makedirs(save_dir, exist_ok=True)

# Save models
for f in ['models/robust_hybrid_best.pt', 'models/teacher_text_best.pt',
          'models/students_distilled_aligned.pt', 'models/baseline_text_dominant.pt']:
    if os.path.exists(f):
        shutil.copy(f, save_dir)
        print(f"Saved: {f}")

# Save logs and figures
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(save_dir, 'logs'), dirs_exist_ok=True)
if os.path.exists('figures'):
    shutil.copytree('figures', os.path.join(save_dir, 'figures'), dirs_exist_ok=True)

print(f"\n✅ All results saved to: {save_dir}")